# 03. Полный пайплайн: модель → микс датасетов → инференс → атомизация → кэш → выборка

В этом ноутбуке показываем **конечный model-first pipeline**:

1. выбираем модель;
2. собираем единый mixed batch из нескольких совместимых датасетов;
3. прогоняем модель и получаем `LayerIORecord`;
4. атомизируем в `SharedSample`;
5. складываем в bounded cache;
6. читаем как `SharedModelDataset`.


In [ ]:
DATA_ROOT = "./data_tutorials"
ENABLED_DATASETS = ["flickr30k", "coco2017", "scene_parse_150"]
DEMO_NUM_SAMPLES = 200
RUN_PREDOWNLOAD = False  # True для явной проверки predownload stage

print("DATA_ROOT:", DATA_ROOT)
print("ENABLED_DATASETS:", ENABLED_DATASETS)
print("DEMO_NUM_SAMPLES:", DEMO_NUM_SAMPLES)


In [ ]:
from __future__ import annotations

import os
import sys
import platform
import time
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import torch
from PIL import Image
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.data_raw.providers.hf import register_all_adapters
from dataset.data_raw.providers.hf.auth import get_hf_token
from dataset.data_raw.registry import create_dataset
from dataset.models.registry import create_model
from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset
from dataset.shared.compatibility_index import CompatibilityIndex


def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        cfg = compose(config_name=cfg_path.stem, overrides=overrides or [])
    return cfg


def resolve_dataset_cfg(cfg, dataset_yaml_name: str):
    data_cfg = cfg.data
    config_dirs = list(data_cfg.get("dataset_config_dirs", ["conf/data/datasets"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{dataset_yaml_name}.yaml"
        if candidate.exists():
            ds_cfg = OmegaConf.load(candidate)
            override_map = cfg.data.get("dataset_overrides", {})
            if dataset_yaml_name in override_map:
                ds_cfg = OmegaConf.merge(ds_cfg, override_map[dataset_yaml_name])
            return ds_cfg
    raise FileNotFoundError(f"Не найден YAML датасета: {dataset_yaml_name}")


def resolve_model_cfg(cfg, model_yaml_name: str):
    models_cfg = cfg.models if "models" in cfg else {}
    config_dirs = list(models_cfg.get("model_config_dirs", ["conf/data/models"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{model_yaml_name}.yaml"
        if candidate.exists():
            return OmegaConf.load(candidate)
    raise FileNotFoundError(f"Не найден YAML модели: {model_yaml_name}")


def show_image_grid(pil_list, n: int = 16, title: str | None = None):
    if not pil_list:
        print("Список изображений пуст")
        return
    images = pil_list[:n]
    cols = min(4, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    idx = 0
    for r in range(rows):
        for c in range(cols):
            ax = axes[r][c]
            ax.axis("off")
            if idx < len(images):
                ax.imshow(images[idx])
                ax.set_title(f"#{idx}")
            idx += 1
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def print_yaml_section(path_to_yaml: str):
    path = repo_root / path_to_yaml
    if not path.exists():
        print(f"Файл не найден: {path}")
        return
    print("=" * 80)
    print(f"RAW YAML: {path}")
    print("=" * 80)
    print(path.read_text(encoding="utf-8"))
    print("=" * 80)
    print("OmegaConf (resolve=False)")
    print("=" * 80)
    obj = OmegaConf.load(path)
    print(OmegaConf.to_yaml(obj, resolve=False))
    print("=" * 80)
    print("OmegaConf (resolve=True, если возможно)")
    print("=" * 80)
    try:
        print(OmegaConf.to_yaml(obj, resolve=True))
    except Exception as exc:
        print(f"Не удалось resolve=True: {exc}")


def ensure_hf_token(cfg):
    token = get_hf_token(cfg)
    if not token:
        raise ValueError(
            "HF token missing in top-level config: set hf.token. "
            "Для gated ресурсов также примите лицензию на странице HF."
        )
    return token


def print_system_info():
    print(f"Python: {platform.python_version()}")
    print(f"Torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")


In [ ]:
print_system_info()

if torch.cuda.device_count() >= 2:
    suggested_train_device = "cuda:0"
    suggested_collector_device = "cuda:1"
    print("Рекомендуемый async режим: train=cuda:0, collector=cuda:1")
else:
    suggested_train_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    suggested_collector_device = None
    print("Fallback interleaved: collector.device=null")


## Quick start (опционально одной ячейкой)

Поставьте `QUICK_START = True` для мини-проверки shared pipeline с маленьким кэшем.


In [ ]:
QUICK_START = False

if QUICK_START:
    qs_overrides = [
        f"data.path={DATA_ROOT}",
        "data.enabled_datasets=[flickr30k,coco2017]",
        "data.dataset_overrides.flickr30k.models=[clip_vit_b32]",
        "data.dataset_overrides.coco2017.models=[clip_vit_b32]",
        "collector.cache.max_items=64",
        "collector.cache.fill_target=64",
        "collector.cache.low_watermark=32",
        "collector.atomization.chunk_rows=64",
        "collector.interleaved.every_n_steps=5",
        "collector.interleaved.burst_jobs=1",
        "collector.device=null",
        "collector.mode=auto",
    ]
    cfg_qs = load_hydra_cfg(overrides=qs_overrides)
    collector_qs = CollectorService(cfg_qs)
    collector_qs.start()

    consumed = 0
    for step in range(50):
        collector_qs.maybe_collect(step)
        item = collector_qs.cache.try_get()
        if item is not None:
            consumed += 1

    print("quick-start consumed:", consumed)
    print("quick-start stats:", collector_qs.stats())
    collector_qs.shutdown()


## Как взаимодействуют конфиги

- `conf/config.yaml`: глобальные настройки (`hf.token`, `data.path`, `train.device`, `collector.*`).
- `conf/data/datasets/*.yaml`: какие модели поддерживает каждый датасет (`models: [...]`).
- `conf/data/models/*.yaml`: как запускать модель (`run_mode`, `batch_size`, `hook_filter`, limits).

**CompatibilityIndex** строит граф: `model -> datasets`.

Далее scheduler выбирает **модель первой**, после чего raw pool собирает **единый mixed batch** из всех совместимых датасетов.


In [ ]:
print_yaml_section("conf/data/datasets/_data_raw_.yaml")
print_yaml_section("conf/data/models/_model_.yaml")


## Минимальная конфигурация эксперимента (overrides)

Зададим маленький кэш, маленький burst и 2 модели с пересечением по датасетам.


In [ ]:
common_overrides = [
    f"data.path={DATA_ROOT}",
    f"data.enabled_datasets=[{','.join(ENABLED_DATASETS)}]",
    "data.dataset_overrides.flickr30k.models=[clip_vit_b32,dinov2_base]",
    "data.dataset_overrides.coco2017.models=[clip_vit_b32]",
    "data.dataset_overrides.scene_parse_150.models=[dinov2_base]",
    "collector.model_burst_jobs=1",
    "collector.dataset_mix_policy=multinomial",
    "collector.mix_cap_per_dataset=0.6",
    "collector.cache.max_items=300",
    "collector.cache.fill_target=300",
    "collector.cache.low_watermark=150",
    "collector.atomization.atom_mode=chunk",
    "collector.atomization.chunk_rows=128",
]

cfg_preview = load_hydra_cfg(overrides=common_overrides)
print("collector config preview:")
print(OmegaConf.to_yaml(cfg_preview.collector, resolve=True))


In [ ]:
index = CompatibilityIndex(cfg_preview)
models = index.get_models()
print("Collectable models:", models)

for m in models:
    ds = index.get_datasets_for_model(m)
    w = index.get_dataset_weights_for_model(m)
    print(f"{m} -> datasets={ds}, weights={w}")


## Async mode (если есть выделенный collector GPU)


In [ ]:
async_stats = None
async_samples = []

if suggested_collector_device is not None and suggested_collector_device != suggested_train_device:
    cfg_async = load_hydra_cfg(overrides=common_overrides + [
        f"train.device={suggested_train_device}",
        f"collector.device={suggested_collector_device}",
        "collector.mode=auto",
    ])

    collector_async = CollectorService(cfg_async)
    if RUN_PREDOWNLOAD:
        collector_async.predownload_models()

    collector_async.start()
    shared_async = SharedModelDataset(collector_async)

    model_counts = Counter()
    dataset_mix = Counter()
    layer_counts = Counter()
    runid_mix = {}

    t0 = time.time()
    while len(async_samples) < min(200, DEMO_NUM_SAMPLES) and (time.time() - t0) < 120:
        sample = collector_async.cache.try_get()
        if sample is None:
            time.sleep(0.1)
            continue
        async_samples.append(sample)
        model_counts[sample.model_name] += 1
        layer_counts[sample.layer_name] += 1

        run_id = sample.meta.get("model_run_id")
        if run_id is not None and run_id not in runid_mix:
            c = Counter()
            for item in sample.meta.get("image_meta", []):
                ds_name = item.get("dataset_name")
                if ds_name:
                    c[ds_name] += 1
            runid_mix[run_id] = dict(c)

        for item in sample.meta.get("image_meta", []):
            ds_name = item.get("dataset_name")
            if ds_name:
                dataset_mix[ds_name] += 1

    async_stats = {
        "cache_size": collector_async.cache.size(),
        "num_samples": len(async_samples),
        "model_counts": dict(model_counts),
        "dataset_mix": dict(dataset_mix),
        "top_layers": layer_counts.most_common(5),
        "dataset_mix_per_job_first10": {k: runid_mix[k] for k in sorted(runid_mix)[:10]},
    }
    collector_async.shutdown()

    print("ASYNC stats:")
    print(async_stats)
else:
    print("Async-режим пропущен: нет выделенного collector GPU")


## Interleaved mode (fallback для single GPU)

Симулируем training loop и вызываем `collector.maybe_collect(step)`.


In [ ]:
cfg_inter = load_hydra_cfg(overrides=common_overrides + [
    f"train.device={suggested_train_device}",
    "collector.device=null",
    "collector.mode=auto",
    "collector.interleaved.every_n_steps=10",
    "collector.interleaved.burst_jobs=1",
])

collector_inter = CollectorService(cfg_inter)
if RUN_PREDOWNLOAD:
    collector_inter.predownload_models()
collector_inter.start()

shared_inter = SharedModelDataset(collector_inter)

model_counts_inter = Counter()
dataset_mix_inter = Counter()
layer_counts_inter = Counter()
job_mix_per_job = {}
last_sample = None

for step in range(200):
    stats_list = collector_inter.maybe_collect(step)
    for st in stats_list:
        key = len(job_mix_per_job)
        job_mix_per_job[key] = dict(st.dataset_mix)

    sample = collector_inter.cache.try_get()
    if sample is None:
        continue

    last_sample = sample
    model_counts_inter[sample.model_name] += 1
    layer_counts_inter[sample.layer_name] += 1
    for item in sample.meta.get("image_meta", []):
        ds_name = item.get("dataset_name")
        if ds_name:
            dataset_mix_inter[ds_name] += 1

collector_inter_stats = {
    "cache_size": collector_inter.cache.size(),
    "num_consumed": int(sum(model_counts_inter.values())),
    "model_counts": dict(model_counts_inter),
    "dataset_mix": dict(dataset_mix_inter),
    "top_layers": layer_counts_inter.most_common(5),
    "dataset_mix_per_job_first10": {k: job_mix_per_job[k] for k in sorted(job_mix_per_job)[:10]},
}

collector_inter.shutdown()
print("INTERLEAVED stats:")
print(collector_inter_stats)


## Инспекция одного `SharedSample`


In [ ]:
if last_sample is None and async_samples:
    last_sample = async_samples[0]

if last_sample is None:
    print("Сэмпл пока не получен")
else:
    print("model_name:", last_sample.model_name)
    print("layer_name:", last_sample.layer_name)
    print("x shape:", tuple(last_sample.x.shape), last_sample.x.dtype, last_sample.x.device)
    print("y shape:", tuple(last_sample.y.shape), last_sample.y.dtype, last_sample.y.device)
    print("meta keys:", list(last_sample.meta.keys()))

    x = last_sample.x
    print("x mean/std:", float(x.mean()), float(x.std()))


## Как расширять систему

### Добавить новый raw датасет

1. Создать YAML на основе `conf/data/datasets/_data_raw_.yaml`.
2. Добавить adapter в `dataset/data_raw/providers/hf/`.
3. Указать поддерживаемые модели в `models: [...]`.

### Добавить новую модель

1. Создать YAML на основе `conf/data/models/_model_.yaml`.
2. При необходимости добавить раннер в `dataset/models/providers/...`.
3. Настроить `run_mode`, `hook_filter`, `limits`.

### Добавить связь dataset<->model

Просто обновить `models: [...]` в dataset YAML.


## Troubleshooting

- **Кэш не заполняется**:
  - проверьте `hf.token`, лицензии gated ресурсов, доступность HF.
  - проверьте `collector.device` и выбранный режим.
- **Слишком медленное переключение моделей**:
  - уменьшите `model_burst_jobs`, `batch_size`.
- **Слишком много атомов за job**:
  - уменьшите `collector.atomization.chunk_rows` и `limits.max_records_per_layer`.
- **IPC overhead в async**:
  - увеличивайте размер атома (chunk mode), не отправляйте слишком мелкие куски.
